# rail3D — the complete pipeline

One notebook for the whole experiment: what the system is, why each design choice was made,
and every command needed to run it end to end.

**How this notebook is built.** Long jobs (dataset generation, the detector sweep) run as
*subprocesses* so the kernel does not accumulate GPU memory and a dropped connection cannot
lose progress — both are resumable, so re-running the cell continues where it stopped.
Everything light (checks, figures, analysis) runs in-kernel so plots appear inline.

**Run order.** Sections 1–3 are checks and context (fast). Section 4 explains the physics.
Sections 5–6 build the dataset. Sections 7–9 train and analyse. Green 'RUN' cells are the
ones that do work; the rest is explanation.

> If you are stepping away during Section 5 (~36 min), prefer the terminal command shown
> there — VS Code disconnects kill notebook cells (though the run is resumable).

## 0. Kernel setup

`RAILDEFECT_DATA_DIR` is read at **import time**, so it must be set before `rail3d` is
imported. If you launched VS Code with `code .` from a shell where you exported it, the
kernel already has it and the line below is a no-op.

In [ ]:
import os
# Set this if the kernel did not inherit it. Windows path with forward slashes.
os.environ.setdefault('RAILDEFECT_DATA_DIR', r'C:/Users/ct2443/Downloads/RailDefect/RailDefect')

import subprocess, sys, json
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

HERE = Path.cwd()
assert (HERE / 'rail3d').is_dir(), f'run this notebook from the rail3D folder (cwd={HERE})'
sys.path.insert(0, str(HERE))

from rail3d import config, data3d, losses3d, mesh3d, optics3d, sections, train3d, viz_setup

PROFILE = 'lab'          # 'laptop' for a small GPU, 'cpu' for no GPU
device = config.get_device(PROFILE)
config.ensure_dirs()

def run(*args, timeout=None, check=True):
    """Run a pipeline script as a subprocess, streaming output live.
    Isolation matters: the child frees its GPU memory on exit, so a long
    generation cannot starve a later training cell in this kernel.

    check=True RAISES on a non-zero exit. This matters in a multi-hour
    sequential run: without it a failed generation just prints a traceback and
    the next cell trains on a partial dataset. Pass check=False where non-zero
    is expected (preflight exits 1 on any 'bad' finding, which is often the
    guard working as intended)."""
    p = subprocess.Popen([sys.executable, *args], cwd=HERE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'{args[0]} exited {rc} — fix this before continuing')
    return rc

def free_gpu():
    """Call between heavy in-kernel phases; the kernel holds tensors alive otherwise."""
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU reserved: {torch.cuda.memory_reserved(device)/1e9:.2f} GB')

def show(name):
    p = config.FIGURE_DIR / name
    display(Image(filename=str(p))) if p.exists() else print(f'{name} not generated yet')

print(f'device      {device}')
print(f'geometry    grid {config.NX}x{config.NY}, H_MS={config.H_MS} mm, '
      f'segment {config.SEG_LEN} mm, mesh lambda/{config.WVL/config.MESH_DS:.0f}')
print(f'classes     {list(config.CLASS_NAMES)}')

## 1. Pre-flight — is this machine consistent?

**RUN THIS FIRST, EVERY TIME.** Every problem hit during development was a consistency
problem rather than a code bug: stale code on one machine, a checkpoint from when there
were 3 classes, a dataset generated at a different plane height, shards silently reused
across runs because the generator skips files that already exist.

`preflight.py` checks all of it in seconds and prints the exact fix for anything wrong.
Exit code 0 means safe to proceed.

In [ ]:
# exit 1 here is usually the guard working: datasets generated before the
# 2026-09-11 shadow decision are refused, not broken. Read the findings.
run('preflight.py', '--profile', PROFILE, check=False)


If it reports **STOP**, the most common causes and fixes:

| finding | why it matters | fix |
|---|---|---|
| behind origin | you would run stale code against fresh data | `git checkout -- rail3D/data/ && git pull` |
| `dataset_config.json` missing | the shards predate provenance recording, so their geometry cannot be verified | generate into a new `--name` root |
| geometry mismatch | fields were generated at a different wavelength / plane height / grid / class list — training on them produces numbers that look fine and mean nothing | generate into a new `--name` root; keep the old set as the record |
| shards written hours apart | the generator **skips existing shards**, so old and new geometry may be mixed (the generator now refuses this outright, but pre-guard roots can still exist) | regenerate into a fresh `--name` root in one go |
| checkpoint class-count / geometry mismatch | it predates a change to `CLASS_NAMES` or the wavelength and cannot be resumed | `rm -rf data/checkpoints/<run>` |

**After the λ=5 migration every λ=8 dataset and checkpoint reports as stale — that is the
guard working.** Nothing is deleted automatically; generate a λ=5 set with `--name` and the
old ones stay on disk as the record.

## 2. What this system is

A **trainable metasurface plus detector array** that identifies rail-head defects from a
single microwave measurement — no digital image is formed. The optics *are* the classifier.

```
horn (140 mm @ 55 deg)
        |  illuminates the rail head
        v
rail surface with a defect       <- physical optics: exact Rayleigh-Sommerfeld surface integral
        |  scatters
        v
measurement plane, 60x30 @ 2.5 mm, 150 mm above the crown
        |  metasurface: a trainable phase map
        v
free-space propagation, 100 mm   <- same exact RS kernel, applied by FFT
        |
        v
detector windows -> 'barcode' -> defect / no-defect + which class
```

The physics is ported from the **experimentally validated** Face3D codebase (a facial-
recognition metasurface at 8 mm), so the solver and propagator are not new code — they are
verified code re-used. The system now runs at **lambda = 5 mm (60 GHz)** as an exact
5/8-scaled replica of that validated scene: every rig length Face3D chose in wavelengths
scales with lambda, so the Fresnel numbers and speckle statistics are preserved, while the
rail and its defects keep their physical sizes — a 2 mm hairline crack is 0.4 lambda now
instead of 0.25 lambda. (The Face3D meta-atom *library* does not transfer across
wavelength, so the fabricable MetaUnit surface is blocked until a 60 GHz library is fitted;
the idealized SLM phase surface is wavelength-agnostic.)

**Why a metasurface at all?** Without one, detector windows just integrate whatever the rail
happens to scatter onto them. The metasurface reshapes the field *before* detection so that
defect-relevant differences land where detectors can see them. Section 9 compares against a
no-metasurface baseline to quantify exactly that.

## 3. The measurement geometry, and why

Two choices here were **measured, not inherited**, and both matter more than they look.
(The measurements below were made at lambda=8 in absolute mm; the current config expresses
them in wavelengths — H_MS = 30 lambda — so the scaled replica reproduces the same physics.
The rig scaled but the defects did not, so `scan_geometry.py` must be RE-RUN at lambda=5
before the full generation to confirm 30 lambda still wins.)

### The plane is dark-field on purpose

The horn illuminates at 55 deg, so the specular lobe off a flat crown lands at
`x = -H*tan(55 deg) = -2.14 H` — far outside the aperture at H = 30 lambda. The system
deliberately collects *off-specular* scatter.

That is not an accident of framing: placing the plane where the lobe **is** captured
(H = 10 lambda) collects **13x more energy** and performs **below chance**. The lambda=8
scan (H in mm at that wavelength; divide by 8 for units of lambda):

| H (mm @ lambda=8) | energy | intact spread | field AUC | det AUC |
|---|---|---|---|---|
| 80 (10λ) | 1.43e-2 | 0.0203 | **0.567** | **0.419** |
| 160 (20λ) | 1.06e-3 | 0.0184 | 0.867 | 0.819 |
| **240 (30λ)** | 5.16e-4 | **0.0052** | **0.899** | 0.845 |
| 320 (40λ) | 3.17e-4 | 0.0083 | 0.890 | 0.842 |
| 480 (60λ) | 1.60e-4 | 0.0022 | 0.869 | 0.887 |

The specular lobe is dominated by the intact rail's mirror reflection: lots of power, almost
no defect information, and it swamps the signal.

**H = 30 lambda was chosen on field AUC** (information physically present at the plane)
rather than det AUC (what a fixed, untrained readout extracts). Training can fix a readout
bottleneck; it cannot recover information that is not there. Beyond 30 lambda the field
grows diffuse and crack speckle washes out — crack field AUC fell 0.803 -> 0.688 — while
det AUC rises only because broader detector footprints are less sensitive to placement
jitter. It also keeps 3x more energy than 60 lambda, which our *relative* noise model does
not penalise but real hardware would.

Re-derive it any time with `run('scan_geometry.py')` (the default candidates are
lambda-multiples derived from the active config).

In [ ]:
run('setup_diagram.py')
show('setup_diagram.png')

Read the middle panel: the metasurface at z=150 mm, detectors at z=250 mm, the horn at
140 mm / 55 deg, and the red x marking where the specular lobe lands — visibly outside the
aperture. The right panel shows the 120 mm rail segment against the 150x75 mm aperture
(the segment now extends past the y-aperture, which is fine — oblique scattering from the
overhang still lands on the plane).

**Why the segment is only 120 mm:** a truncation study at lambda=8 showed 120 mm preserves
every class's defect signature to >= 0.997 cosine versus a 240 mm reference while being
**3.1x faster** to generate. Truncation shifts the intact field ~5%, but that is common
mode — intact and defect samples share the segment, so it cancels in the comparison. At
lambda=5 the illuminated footprint shrinks with lambda, so 120 mm is *safer* than when it
was measured. It is deliberately NOT lambda-scaled: the defects it must contain (50 mm
cracks, wear envelopes) are physical.

## 4. The defect model

Defects are **per-point depth fields** on the rail surface:

$$\\text{displaced}(s, y) = \\text{intact}(s) - d(s, y)\\,\\hat{n}(s)$$

where `s` runs across the head, `y` along the rail, and `d >= 0` is the depth in mm. An
earlier version blended whole cross-sections per slice, which could only make defects that
were uniform across the head — it could not represent a crack running *along* the rail, or a
compact dent.

| class | footprint | depth | where |
|---|---|---|---|
| `crack` | line divot 10-50 mm long x 2-5 mm wide; longitudinal / transverse / oblique; 30% chance of 2-3 parallel | 2-10 mm | running band + gauge corner |
| `dent` | 2D super-Gaussian 10-30 x 10-30 mm; 20% chance of a pit chain | 1.5-8 mm | running band |
| `wear` | worn cross-section over a 300-900 mm envelope, i.e. the whole segment | 2-8 mm | horn-facing shoulder |
| `shell` | ragged Fourier-modulated ellipse 8-20 mm; 30% chance of a second lobe | 1-5 mm | horn-facing shoulder |

Baselines come from laser-scan measurements (Ye et al. 2018 Table 1; Ye et al. 2023 Fig 9);
the ranges above are the operating ranges for this system, widened from those. Depths are
**sampled**, never clipped — clipping raw CSV depths once pinned 66% of cracks at exactly the
cap, destroying depth diversity.

`y0` (along-track position) is confined to +-10 mm because the sensor rides the train: every
defect passes through the beam centre at some frame. The across-head position `s0` stays
broadly sampled — the train cannot move the sensor sideways.

In [ ]:
viz_setup.mesh_review_figure(save_path=config.FIGURE_DIR / 'mesh_review.png')
plt.show()

## 5. Physics verification

Eleven gates, all of which must pass before any result is trustworthy. This is the part
that makes the numbers mean something. V0-V4 are seconds-scale and already green at
lambda=5; V5-V8 re-run here on the lab GPU (their quoted numbers are the lambda=8
baselines until this cell has run).

| gate | what it proves |
|---|---|
| V0 | the defect geometry does what it claims (orientations, band confinement, seed reproducibility, resolution independence) |
| V0b | redundancy pruning keeps a unique detector where variance keeps duplicates |
| V0c | the staleness guards guard: lattice in-aperture, capture clamp, dataset/root/checkpoint refusals |
| V1 | the chunked/batched solver equals the verbatim Face3D integral to ~1e-7 |
| V2 | the FFT propagator equals the verified conv2d kernel to ~1e-6 (~1000x faster) |
| V3 | an independent angular-spectrum propagator agrees to 0.10% — identical to the lambda=8 value, confirming the scaled replica |
| V4 | specular centroid on axis, energy conserved, mesh normals outward |
| V5 | the 3D solver reproduces the established **2D** pipeline on a uniform rail (r = 0.976 at lambda=8; re-measured here) |
| V6 | ray-cast shadowing is real, and free of grazing-ray artifacts (hairlines are 0.4 lambda now — if the resolved-occluder bound trips, that is a physics finding to report, not a bug) |
| V7 | the generation mesh is fine enough (defect-signal cosine vs a 2x finer mesh, judged at the barcode level through the fixed 130-window probe) |
| V8 | training runs end to end: separation grows, 130->8 pruning, detectors genuinely move, no collapse, a killed run resumes **bit-identically**, and the stale-checkpoint refusal fires |

`lab_report.py` runs all of them plus a smoke dataset and writes a paste-able summary.
SETUP_LAB §5 has the field-by-field guide for reading the V8 numbers.

In [ ]:
run('lab_report.py', '--profile', PROFILE)

## 5b. Which modelling choices actually change the answer?

Every gate above asks "is this implementation correct". This section asks the
different question a reviewer will ask: **how much does each modelling choice
move the predicted wavefront**, and which ones are worth defending.

`compare_wavefronts.py` solves ONE physical rail sample many ways and puts the
predicted complex field at the measurement plane side by side:

| version | the choice it isolates |
|---|---|
| `lam5` vs `lam8` | the wavelength migration — same physical rail, each at its own scaled geometry. Both planes are 60x30 over the same angular extent, so the arrays are directly comparable. |
| `psi0 / psi1 / psi1+psi2 / tot` | how much each physics term contributes (`tot` is what the dataset stores) |
| `raycast 3.0` vs `raycast 0.125` vs `no shadow` | the ray-cast self-shadow guard, `config.SHADOW_MIN_T` |
| `mesh lambda/16` | mesh convergence, the same question V7 gates |

Three metrics, because field-level and detector-level disagree in an important
way: **relative L2** and **complex correlation** on the raw field, and
**barcode cosine** through the detector windows. README finding 2 is the reason
both are reported — raw complex-field L2 never converges (glint speckle), so
the barcode is the level fidelity is actually judged at. A version can move the
field several percent and the barcode almost not at all; that is a real result,
not a contradiction.

The same script exports an **FDTD-ready case** and accepts external solver
results back, which is how the full-wave comparison joins these figures.

In [ ]:
# ~1-2 min on the 5090 at production geometry; add --seg 30 on a small GPU.
# --export-case also writes the geometry + source spec for an external solver.
run('compare_wavefronts.py', '--profile', PROFILE,
    '--export-case', 'data/generated/fdtd_case')
show('wavefront_cuts.png')

**Reading the four panels.** Top row is each version's amplitude and raw
phase; the raw phase is dominated by the common propagation ramp (tens of
radians across the aperture), which is exactly why the bottom row plots
everything *relative to the reference* — that is where the versions actually
separate. A curve marked `[identical to ref]` in the legend sits exactly on the
reference, which is a finding in itself.

The maps and the metric bars give the same story two other ways:

In [ ]:
show('wavefront_maps.png')
show('wavefront_metrics.png')

### The ray-cast shadow guard (`config.SHADOW_MIN_T`)

`min_t` ignores any occluder found within that distance *along the ray* from
the face it started at. It exists because facet chords sag inside the true
convex surface, so horizon-grazing rays clip their own neighbours — without a
guard ~4% of terminator faces are falsely blocked and the field moves 30-60%
(README finding 3). The question is how big it should be, and the two
populations it has to separate were measured directly:

| population | where it lives |
|---|---|
| discretization artifact (intact rail — a convex rail cannot shadow itself) | `t <= 0.021 mm` at lambda=5, `<= 0.037 mm` at lambda=8 — the chord sagitta `d^2/(8R)`, about 1/100 of a facet |
| real crater-wall occlusion (cracks, dents, shells) | `t >= 0.3 mm`, out to ~9 mm |

They are cleanly separated, and the historical `min_t = 3.0 mm` sits far above
the gap — so it discards real self-shadowing along with the artifact. In the
wavefront comparison above, `raycast 3.0` comes out **identical** to no shadow
at all for a crack, while `raycast 0.125` moves the field by several percent.

`--min-t` sweeps it at the field level, which is what should decide the value:
the artifact metric (on augmented INTACT meshes, where every bit of change is
error) must stay flat and under V6's 3% bound, and the right choice is the
SMALLEST value that keeps it there.

In [ ]:
# SETTLED 2026-09-11 — this sweep is kept for reproduction, not decision.
# The result: normal_offset >= 0.05 drives the intact artifact to EXACTLY 0.0
# at every min_t, while real crack shadowing stays at 5-11%; min_t alone can
# never separate the two, because the self-hit is a perpendicular problem and
# min_t biases along the ray. Shipped: min_t 0.05, offset 0.3.
# README finding 3 has the full table. Re-running overwrites the good
# V6b_guard_sweep entry in verification_report.json with a narrower one, so
# leave this off unless you are deliberately re-deciding.
RERUN_SHADOW_SWEEP = False

if RERUN_SHADOW_SWEEP:
    run('validation_3d.py', '--min-t', '0.05', '0.5', '1.5', '3.0',
        '--normal-offset', '0', '0.05', '0.15', '0.3', '0.6',
        '--crack-idx', '0', '1', '17')
    show('v6b_min_t_sweep.png')
else:
    print('skipped — settled at min_t 0.05 / offset 0.3; see README finding 3')


**The default is deliberately still 3.0 mm.** Lowering it changes the
physics, so `SHADOW_MIN_T` is a provenance key: datasets generated on either
side of a change are not comparable, and every existing dataset would be
refused. Decide from the sweep above, then set it in `config.py` and regenerate
into a NEW `--name` root.

### Comparing against a full-wave (FDTD / MoM) reference

`--export-case` wrote `data/generated/fdtd_case/`: the rail surface as OBJ (mm,
crown at z=0), and `case.json` with the frequency, horn geometry and incidence,
and the exact cell-centred observation grid. Run that same problem in an
external solver, save the complex field on the same grid, and it joins every
figure and metric here:

```
python compare_wavefronts.py --external fdtd_result.npz
```

Expect PO and full-wave to disagree in specific, predictable places — those are
the interesting slides, not failures to hide: grazing/terminator faces (this PO
formulation applies no receive-cosine, README finding 4), diffraction from the
rail's cut ends, scattering beyond the second bounce, and features at or below
the wavelength (hairline cracks). A practical note: a full FDTD of the whole
120 mm rail plus a 150 mm standoff at 60 GHz is a very large domain, so it is
usually better to validate on a *canonical* case first — a flat plate, then a
plate with one crack — where the PO-vs-FDTD error can be measured cheaply and
then carried over as a calibration.

## 6. Generate the dataset

Estimated **~1.5-2 h** on the 5090 for 4 classes x 5000 + 512 intact: the lambda=8 run
measured 9.47 samples/s (~36 min), and lambda/8 facets at lambda=5 mean (8/5)^2 = 2.56x
the face count. The smoke run in Section 5 measures the true samples/s — trust that
extrapolation.

**Stale roots refuse themselves.** The generator skips shards that already exist — which
is what makes it resumable — and now REFUSES a root whose recorded geometry differs from
the current config (`check_generation_root`), so old and new geometry can no longer mix.
Use `--name` for a fresh root and keep old datasets as the record.

Per sample it: builds the swept mesh at lambda/8, ray-casts shadowing against a lambda/2
occluder, evaluates the exact RS surface integral to the measurement plane for the single
bounce (psi1) and the double bounce (psi2), and stores both. The direct horn term (psi0) is
identical for every sample and is cached once.

> If stepping away, run this in a terminal instead:
> `python generate_dataset_3d.py --profile lab --name L5_H150_v1 --note "lambda=5, H=150"`

### The one-command path

`run_stage.py` does everything in this section and the next two — generate, gate, train, analyse — headlessly and resumably, using the same `config.STAGES` values as the cells below. It **refuses** rather than warns: a short dataset, a geometry mismatch, a non-finite field or a checkpoint from a different schedule all stop the run before the next long step starts.

Run the cell below and skip to section 9, or work through the cells individually if you want to watch a particular step. Generation dominates (~1.8 h for prelim); training is ~3 minutes.


In [ ]:
# The whole chain. Comment out to step through the cells below instead.
# --bundle zips the files worth sending for review.
run('run_stage.py', '--stage', STAGE, '--bundle')


In [ ]:
# What already exists? preflight also lists every dataset with its date.
run('generate_dataset_3d.py', '--profile', PROFILE, '--status')

In [ ]:
# --name keeps datasets side by side instead of overwriting; --note is stored
# in dataset_config.json alongside the geometry, creation time and git commit.
#
# Sizes AND the training schedule live in config.STAGES, shared with
# run_stage.py so the notebook and the script cannot drift apart. See
# README finding 21 for why the schedule is stage-dependent.
STAGE = 'prelim'                       # 'prelim' | 'full'

_s = config.STAGES[STAGE]
TAG = f'l5_{STAGE}'
DATASET = f'data/generated/L5_{STAGE}'
print(f"{STAGE}: {_s['limit']}/class + {_s['intact']} intact -> {DATASET}")
print(f"schedule: n_epoch={_s['n_epoch']}, tau_anneal_end={_s['tau_anneal_end']}, "
      f"prune {_s['prune_start']}-{_s['prune_end']}")


**After a kernel restart, stop here.** The cell above re-establishes `STAGE`, `TAG`, `DATASET` and is instant. The cell below re-invokes the generator — harmless if the dataset is complete (shards are skipped), but `run('generate_dataset_3d.py', '--status')` above tells you that faster.


In [ ]:
run('generate_dataset_3d.py', '--profile', PROFILE,
    '--name', Path(DATASET).name,
    '--limit', str(_s['limit']), '--intact', str(_s['intact']),
    '--note', f"lambda=5, H=150, 4 classes, {STAGE}")


### Check the data before training on it

In [ ]:
run('inspect_dataset.py', '--root', DATASET)
show('dataset_review.png'); show('dataset_meta.png')

Column 1-2 show each stored sample's geometry **re-derived from its seed**, proving the
shard matches the intended defect. Column 3 is the stored field. Column 4 is the mean
defect-minus-intact intensity — the signature the metasurface has to exploit.

## 7. The objective

The first full run scored a detection AUC of 0.75-0.90 but a **false-alarm rate near 0.5**.
The cause was structural, not a tuning problem: with the required +-4 mm placement
augmentation, the *intact* barcode distribution spreads about as much as the defect signal,
so a fixed absolute margin (the 2D pipeline's approach, valid against a single fixed
reference) is ill-posed. Pass rate and false alarm rose together.

The objective is now **ranking-based**:

| term | purpose |
|---|---|
| `rank` | pairwise hinge over all (defect, intact) pairs — a differentiable AUC surrogate |
| `hardest` | the same hinge on the worst 10% of pairs |
| `intact` | pulls the intact cluster tight |
| `class` | cross-entropy on normalised barcodes |
| `power` | keeps light on the detectors |
| `centroid` | pushes the four class centroids apart |
| `tv` | smoothness on the metasurface map (fabricability) |

Barcodes are per-sample L2-normalised, which removes the global-amplitude component of
placement jitter, and the operating threshold is **calibrated** to a target false-positive
rate on the *validation* intact spread — never on test. Checkpoint selection uses AUC plus
class accuracy, both threshold-independent.

`objective='margin'` restores the old loss exactly, and V8 keeps it running as a regression
guard.

## 8. Train

### How the detectors are optimised

Positions are **trained, not swept**: `SoftDetector2D` holds the window centres as an
`nn.Parameter` behind sigmoid-edged (differentiable) masks, so gradients move them.
Redundancy pruning — each detector valued by its unique contribution,
`std x (1 - max|corr| to survivors)` — runs *inside* the same run. With the default
`TrainConfig` schedule one training run does:

| epochs | what happens |
|---|---|
| 0-60 | full 130-window dense start, soft masks, phase + positions training |
| 60-150 | pruning window: geometric reduction to the final count (keep 0.75/step); power floor recomputed at each prune |
| 150-250 | tau anneal completes, masks sharpen to the half-pixel floor |
| 250-1200 | stationary fine-tune at the final count |

All reported metrics use **hard** binary windows, never the soft training masks.

**Why 1200 epochs:** the anneal and pruning make the objective non-stationary for ~250
epochs. A 400-epoch run left only ~150 stationary epochs to settle ~1800 metasurface
parameters. Epochs cost ~0.5 s, so there is no reason to economise. `train()` reports where
the best epoch landed and warns if the model was still improving at the end.

In [ ]:
# EPOCHS ARE NOT OPTIMIZER STEPS, and the difference is a 10-hour trap.
# An epoch is a full pass over the training split, so on the prelim set it is
# 7 steps (16 on the full set). But every schedule constant -- tau_anneal_end,
# prune_start/end -- is expressed in EPOCHS and was tuned on the smoke set,
# where train is a single batch and 1 epoch == 1 step. n_epoch=1200 means 1200
# steps there and 8400 here: ~10 h instead of the intended budget.
# config.STAGES holds values that reproduce the intended STEP budget per stage;
# README finding 21 has the arithmetic.
cfg_slm = train3d.TrainConfig(
    run_name=f'ms3d_slm_{TAG}', surface='slm',
    batch_size=config.PROFILES[PROFILE].train_batch,
    data_root=DATASET,
    **{k: _s[k] for k in ('n_epoch', 'tau_anneal_end', 'prune_start', 'prune_end')})
# the banner prints the real step budget; epoch 1 prints a MEASURED ETA
hist_slm = train3d.train(cfg_slm, device=device)


In [ ]:
def plot_history(hist, title):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    ax[0].plot([t['loss'] for t in hist['train']]); ax[0].set(title='train loss', xlabel='epoch')
    ax[1].plot([v['auc'] for v in hist['val']], label='AUC')
    ax[1].plot([v['pass_rate'] for v in hist['val']], label='pass@cal')
    ax[1].plot([v['false_alarm'] for v in hist['val']], label='false alarm')
    ax[1].plot([v['class_acc'] for v in hist['val']], label='class acc')
    for p in hist['prune_epochs']: ax[1].axvline(p['epoch'], color='gray', lw=0.5, alpha=0.5)
    ax[1].legend(fontsize=8); ax[1].set(title=f'{title} validation (gray: prunes)', xlabel='epoch')
    ax[2].plot([v['gap_mean'] for v in hist['val']], label='defect gap mean')
    ax[2].plot([v['gap0_mean'] for v in hist['val']], label='intact gap mean')
    ax[2].plot([v['threshold'] for v in hist['val']], 'r--', lw=0.8, label='calibrated thr')
    ax[2].legend(fontsize=8); ax[2].set(title='barcode gaps', xlabel='epoch')
    fig.tight_layout(); plt.show()
plot_history(hist_slm, 'SLM')

### The fabricable version — BLOCKED at lambda=5

`MetaUnitSoft` replaces the idealised phase mask with the **real meta-atom**
parameterisation: the trainable variable is a pillar-width map, and amplitude and phase
come from per-pixel polynomial fits to a simulated meta-atom library. That library was
fitted at **8 mm**; the fits do not transfer across wavelength, and the [1, 3.8] mm pillar
widths physically cannot fit the 2.5 mm unit cell at lambda=5 — so `surface='metaunit'`
**raises by design** until a 60 GHz library is fitted. The cell below demonstrates the
refusal; re-enable it once a new library lands in `rail3d/library_*.npy`.

In [ ]:
from dataclasses import replace
# Blocked at lambda=5 (8 mm meta-atom library) — this cell PRINTS the refusal
# rather than training. Restore the train() call when a 60 GHz library exists.
cfg_mu = replace(cfg_slm, run_name=f'ms3d_metaunit_{TAG}', surface='metaunit')
try:
    hist_mu = train3d.train(cfg_mu, device=device)
    plot_history(hist_mu, 'MetaUnit')
except RuntimeError as err:
    print(f'metaunit blocked (expected at lambda=5):\n{err}')

## 9. Results — and where it fails

Aggregate AUC says how well the system does; the analysis below says **which defects it
misses**, by joining each test sample's outcome to the defect parameters stored with it.

**Expect crack to be the limiting class.** It entered training at field AUC 0.803 but det AUC
0.670 — the largest gap between information present and information extracted. Cracks affect
~0.2% of the illuminated surface versus ~8% for wear.

In [ ]:
free_gpu()
run('analyze_results.py', '--run-name', f'ms3d_slm_{TAG}',
    '--data-root', DATASET)
show('analysis_parameters.png'); show('analysis_performance.png')
show('analysis_failures.png')

# Two things to read first, both from READING_RESULTS.md:
#  * crack recall -- the known structural risk. Crack has a LARGER mean signal
#    than shell (10.4% vs 8.3% of peak) but a LOWER untrained AUC (0.736 vs
#    0.817), because its signature DIRECTION rotates with theta over -69..90
#    deg. If recall is poor the remedy is orientation-aware capacity, not more
#    illumination and not a bigger aperture.
#  * capture_frac vs n_det/130 (= 0.0615 at 8 detectors). Above it means the
#    metasurface is concentrating light onto the survivors; sitting on it means
#    the capture term is idle.

In [ ]:
import json
from rail3d import train3d

ana = json.loads((config.GENERATED_DIR / 'analysis.json').read_text())

# WHICH MODEL WAS ANALYSED? analyze_results uses --which best, and "best" is
# only meaningful if that checkpoint met the detector budget. score = auc +
# class_acc and both rise with MORE detectors, so before README finding 22 the
# argmax reliably picked a pre-prune epoch and every number below described the
# DENSE array instead of the pruned system.
ck = torch.load(config.CHECKPOINT_DIR / f'ms3d_slm_{TAG}' / 'best.pt',
                map_location='cpu', weights_only=False)
nd, ep = ck['n_det'], ck['epoch']
ok = nd == config.N_DET_FINAL
print(f"best checkpoint: epoch {ep}, n_det {nd} (target {config.N_DET_FINAL})"
      f"  {'OK' if ok else '<-- NOT the designed system; numbers below are the dense array'}")
print(f"threshold {ana['threshold']:.4f}  (calibrated at target FPR on validation intact)\n")

print(f"{'class':7s} {'n':>5s} {'recall':>7s} {'clsacc':>7s} {'AUC':>6s}   strongest parameter dependence")
for cls, d in ana['classes'].items():
    pc = d.get('param_corr', {})
    top = max(pc.items(), key=lambda kv: abs(kv[1])) if pc else ('-', 0.0)
    print(f"{cls:7s} {d['n']:5d} {d['recall']:7.3f} {d['class_acc']:7.3f} "
          f"{d['auc']:6.3f}   {top[0]} {top[1]:+.2f}")

print("\nHow to read these:")
print("  recall  = detection at the CALIBRATED threshold. Low recall with high")
print("            AUC is an operating-point problem, not a separability one:")
print("            the ROC is shallow near the low-FPR corner.")
print("  clsacc  = 4-way classification. Read it WITH the confusion matrix --")
print("            a class that collapses into another can still score high on")
print("            the class it collapses into.")
print("  param_corr is LINEAR. A peaked dependence (crack vs theta) reads ~0;")
print("            trust analysis_parameters.png over the correlation there.")


### How few detectors are enough?

Each detector is a physical receiver, so the count is a hardware cost. Training starts from
a **dense 13x10 = 130-detector lattice** (92% plane coverage) rather than a sparse grid,
so pruning selects from a rich candidate set. The 13x10 is the **tiling bound**
`floor(aperture / window)`, derived in config — window-sized pitch is the densest USEFUL
start, since the field's speckle grain (lambda*H/D = 9.9 x 6.2 mm, D = the ~76 mm illuminated
head width) is comparable to a window -- so the window is the binding scale for spacing and
anything denser only creates duplicates. The aperture holds ~182 independent speckle cells,
so 130 windows sample near the information limit (README finding 19).

| layout | detectors | pitch | window gap | coverage |
|---|---|---|---|---|
| Face3D 6x6 (lambda=8) | 36 | 48 x 48 mm | +29.8 / +36.8 mm | 7.2% |
| rail3D old 6x3 (lambda=8) | 18 | 36 x 36 mm | +17.8 / +24.8 mm | 12.7% |
| **rail3D 13x10 (lambda=5, now)** | 130 | 10.7 x 6.8 mm | -0.66 / -0.18 mm | **92%** |

A dense start only works with the **redundancy** pruning criterion. Overlapping windows see
nearly the same light and so have nearly the same variance — Face3D's variance ranking cannot
distinguish a duplicate from a uniquely informative detector. In a controlled test with three
bright duplicates and one quiet unique detector, pruning to 3 by variance keeps two duplicates
and **discards the unique signal**; the redundancy criterion (value = std x (1 - max|corr|))
keeps the unique one. Empirically it also ends with detectors about twice as far apart.

Positions themselves are **trained**, not swept (see the schedule above). This sweep varies
the final *count*. Add `--dist 80 100 120` to sweep the metasurface-to-detector distance too
(config default is 20 lambda = 100 mm), and `--prune-criterion variance` to compare against
the old behaviour.

In [ ]:
# One full training run per count -- which used to be the reason this was
# gated off. Measured 2026-09-11: a prelim training run is ~3 minutes, so four
# counts is ~12 minutes. Left on.
run('sweep_detectors.py', '--counts', '4', '6', '8', '10',
    '--epochs', str(config.STAGES[STAGE]['n_epoch']),
    '--data-root', DATASET)
show('detector_sweep.png')


### No-metasurface baseline

The control: identical optics with the metasurface replaced by identity. The difference is
what the metasurface is worth.

In [ ]:
# inherits n_epoch and the schedule from cfg_slm above
cfg_none = replace(cfg_slm, run_name=f'ms3d_none_{TAG}', surface='none')
hist_none = train3d.train(cfg_none, device=device)

import pandas as pd
rows = []
for name, surf in [('SLM', 'slm'), ('MetaUnit', 'metaunit'), ('no MS', 'none')]:
    cfg = replace(cfg_slm, run_name=f'ms3d_{surf}_{TAG}', surface=surf)
    try:
        m, _ = train3d.load_trained(cfg, device, 'best')
        data = train3d.load_all_data(cfg, device)
        ev = train3d.full_evaluation(m, data, cfg)
        rows.append({'model': name, 'AUC': ev['test']['auc'],
                     'class acc': ev['test']['class_acc'],
                     'pass@cal': ev['test']['pass_rate'],
                     'false alarm': ev['test']['false_alarm'],
                     'TPR@1%FPR': ev['roc']['tpr_at_1pct_fpr']})
    except FileNotFoundError:
        print(f'{name}: not trained yet'
              + (' (blocked at lambda=5 — 8 mm meta-atom library)' if surf == 'metaunit' else ''))
pd.DataFrame(rows).set_index('model').round(4) if rows else None

## 10. What to trust, and what to watch

**Trust:** the solver and propagator are verified against the Face3D code they came from
(1e-7, 1e-6) and against the established 2D pipeline (r = 0.976, measured at lambda=8 and
re-measured by Section 5 here). The lambda=5 scene is an exact 5/8-scaled replica of that
validated geometry — V3's angular-spectrum cross-check returns the identical relative error
at both wavelengths, to 6 significant figures. Every dataset records the 36 geometry
constants it was built with; training, the generator and checkpoint resume all REFUSE a
mismatch. A killed training run resumes bit-identically.

**Watch:**
- **Crack recall** — the hardest class by a wide margin, though lambda=5 helps it most
  (a 2 mm hairline is 0.4 lambda now instead of 0.25). `analysis_parameters.png` shows
  whether the misses are the shallow, the short, or a particular orientation.
- **Wear being too easy** — it affects ~8% of the surface and separates almost trivially; a
  near-perfect wear column in the confusion matrix is expected, not a sign of overfitting.
- **`mean |corr|` in the detector sweep** — if it stays high as the count falls, pruning
  kept duplicates rather than complementary detectors.
- **The noise model is relative** (multiplicative + additive, scaled to signal), so it does
  not penalise absolute energy loss. Configurations that collect less light look better here
  than they would on hardware — this is why H was chosen at 30 lambda rather than 60.
- **Ray-cast shadowing uses a lambda/2 occluder.** It measured a real 4.3% effect at
  lambda=8 and should grow at lambda=5 (defects are relatively larger now); V6's
  resolved-occluder control is the check.

Full detail: `README.md` section 6 ('hard-won findings'), section 8 (the consolidated list of
simplifications and limitations), and `SETUP_LAB.md`.